# Web Crawling for RAG — Crawl4AI vs. Firecrawl vs. Scrapy vs. fastCRW

Every loader notebook in this course (`l22_document_loaders.ipynb`, `l22_chunking_strategies.ipynb`)
starts from a file that's already sitting in `data/`. Crawling is the step *before* that: given
just a starting URL, discover and fetch the pages of a live website so there's something to load,
chunk, and embed in the first place.

Four tools, by RAG suitability:

| Tool | Best for | RAG suitability | Pick |
|---|---|---|---|
| **Crawl4AI** | Python + AI/RAG + dynamic websites | ⭐⭐⭐⭐⭐ | #1 |
| **Firecrawl** | Production-grade crawling + clean Markdown | ⭐⭐⭐⭐⭐ | #2 |
| **Scrapy** | Large-scale traditional crawling | ⭐⭐⭐⭐ | #3 |
| **fastCRW** | Hosted, AI-agent-oriented crawl/scrape/extract API | ⭐⭐⭐⭐ | added below |

`fastCRW` (`fastcrw.com`, Python SDK on PyPI as `crw`) is a newer entrant in the same space as
Firecrawl: a managed cloud API purpose-built for AI agents, with a free tier (500 credits, no
card required) and an open-source (AGPL-3.0) self-hostable option. It wasn't part of the
original three-way ranking, so it's added here as a fourth data point rather than forced into
the #1-#3 slots above.

**Target site:** HDFC Bank. The well-known `www.hdfcbank.com` currently returns a broken
Cloudflare "DNS points to prohibited IP" error, so this notebook uses the bank's actual working
site, **`https://www.hdfc.bank.in`** (India's bank-only `.bank.in` domain; `hdfcbank.com`
redirects here). Its `robots.txt` explicitly allows crawling (`User-agent: * → Allow: /`) and
even explicitly whitelists AI bots (`ClaudeBot`, `GPTBot`, `PerplexityBot`, ...).

**Ground rules followed throughout this notebook:**
- Same-domain only (`www.hdfc.bank.in`), starting from the public homepage.
- A small page budget — a handful of pages per tool, not a full site crawl.
- `robots.txt` is obeyed everywhere it can be (Scrapy's `ROBOTSTXT_OBEY`; Crawl4AI/Firecrawl
  already permitted for all crawlers per the site's own policy).
- A polite delay between requests — no hammering.
- GET requests on public informational pages only — never NetBanking, login, or forms.
- A descriptive `User-Agent` identifying this as an educational crawl.


In [1]:
# pip install crawl4ai scrapy firecrawl-py   (already added via `uv add`)
# Playwright's Chromium binary was already cached in this environment, so Crawl4AI
# (which drives Playwright under the hood) needed no browser download.

from dotenv import load_dotenv
from langchain_core.documents import Document
from urllib.parse import urlparse

load_dotenv()

BASE_URL = "https://www.hdfc.bank.in/"
ALLOWED_HOST = "www.hdfc.bank.in"
MAX_PAGES = 20  # small, polite budget — shared across all three tools
USER_AGENT = "genai-class-educational-crawler"

def same_domain(url: str) -> bool:
    return urlparse(url).netloc.lower() == ALLOWED_HOST


## 1. Crawl4AI — the #1 pick

Crawl4AI is built specifically for feeding LLM pipelines: it renders pages with Playwright (so
JS-heavy sites work), and converts HTML to Markdown with a **content-pruning filter** that
strips navigation/boilerplate — exactly the noise a naive `BeautifulSoup` scrape would keep.


### Single page — raw Markdown vs. pruned Markdown

The difference between `raw_markdown` and `fit_markdown` is the whole pitch for Crawl4AI in a
RAG context: the raw conversion keeps every nav link and menu item; the pruned version keeps
only the page's actual content.


In [2]:
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator
from crawl4ai.content_filter_strategy import PruningContentFilter

md_generator = DefaultMarkdownGenerator(
    content_filter=PruningContentFilter(threshold=0.48, threshold_type="fixed")
)
single_page_config = CrawlerRunConfig(markdown_generator=md_generator, user_agent=USER_AGENT)

async with AsyncWebCrawler() as crawler:
    homepage = await crawler.arun(url=BASE_URL, config=single_page_config)

print(f"raw markdown:    {len(homepage.markdown.raw_markdown):,} chars")
print(f"pruned markdown: {len(homepage.markdown.fit_markdown):,} chars")


NotImplementedError: 

In [3]:
homepage

CrawlResultContainer([CrawlResult(url='https://www.hdfc.bank.in/', html='<!DOCTYPE html><html lang="en-IN"><head>\n\n  <script>\n    (function() {\n\n        if (typeof Swiper === "undefined") return;\n\n        const wrapper = document.querySelector(\'.pageWrapper\');\n        if (!wrapper) return;\n\n        /* CSS */\n\n        const style = document.createElement(\'style\');\n\n        style.innerHTML = `\n.personalNavWrap{display:none}\n\n@media(max-width:991px){\n\n.personalNavWrap{\ndisplay:block;\nbackground:#e6f1fb\n}\n\n.personalNav{\npadding:0 40px;\nposition:relative\n}\n\n.personalNav .swiper-slide{\nfont-size:.75rem;\nfont-family:OpenSans_Semibold;\ncolor:#004c8f;\npadding:10px 0;\nmargin:0 8px;\nwhite-space:nowrap;\ncursor:pointer;\nwidth:auto;\nposition:relative\n}\n\n.personalNav .swiper-slide.active{\nfont-family:OpenSans_Bold\n}\n\n.personalNav .swiper-slide.active:after{\ncontent:"";\nposition:absolute;\nbottom:8px;\nleft:0;\nright:0;\nheight:1px;\nbackground:#004c8

In [4]:
print(homepage.markdown.fit_markdown)


Skip To Main Content 
Dream Bigger.Grow Further.
Check out our National Anthem that isn't just sung - it is signed!
Select from the below options: Products Services
Select Product Type 
  * [Credit Card Services](javascript:void\(0\); "Credit Card Services")
  * [Social Security Schemes](javascript:void\(0\); "Social Security Schemes")


Select Product


## HDFC Bank Mobile Banking App
What's more! You can do whatever banking need you have on our all-new mobile banking app!
[For Android (opens in a new tab)](https://play.google.com/store/apps/details?id=com.hdfcbank.android.now "For Android") [For IOS (opens in a new tab)](https://apps.apple.com/us/app/id6504402552 "For IOS")
ASK
EVA
## Banking Solutions tailor-made for you
  1. ### Trending
  2. ### Accounts
  3. ### Deposits
  4. ### Cards
  5. ### Loans
  6. ### Insurance
  7. ### Investments


###  Loan On Credit Card 
Get instant cash at a low interest rate, anytime
[Check Eligibility (opens in a new tab)](https://applyonline.hdfc

### Multi-page — BFS crawl a few links deep

Crawl4AI has a built-in deep-crawl strategy so we don't have to hand-write link discovery.
`DomainFilter` keeps the crawl on `hdfc.bank.in`, and `same_domain()` (defined above) tightens
that further to exactly `www.hdfc.bank.in` — belt-and-suspenders so nothing wanders onto an
unrelated subdomain.


In [11]:
from crawl4ai.deep_crawling import BFSDeepCrawlStrategy
from crawl4ai.deep_crawling.filters import FilterChain, DomainFilter

deep_config = CrawlerRunConfig(
    deep_crawl_strategy=BFSDeepCrawlStrategy(
        max_depth=3,
        max_pages=MAX_PAGES,
        filter_chain=FilterChain([DomainFilter(allowed_domains=["hdfc.bank.in"])]),
    ),
    markdown_generator=md_generator,
    user_agent=USER_AGENT,
)

async with AsyncWebCrawler() as crawler:
    deep_results = await crawler.arun(url=BASE_URL, config=deep_config)

crawl4ai_pages = {}
for r in deep_results:
    if r.success and same_domain(r.url) and r.url not in crawl4ai_pages:
        crawl4ai_pages[r.url] = r

print(f"{len(crawl4ai_pages)} unique pages kept")
for url in crawl4ai_pages:
    print(" -", url)


[INIT].... → Crawl4AI 0.9.2 

[FETCH]... ↓ https://www.hdfc.bank.in/                                                                            | ✓ | ⏱: 0.80s 

[SCRAPE].. ◆ https://www.hdfc.bank.in/                                                                            | ✓ | ⏱: 0.66s 

[COMPLETE] ● https://www.hdfc.bank.in/                                                                            | ✓ | ⏱: 1.47s 

[FETCH]... ↓ https://www.hdfc.bank.in/                                                                            | ✓ | ⏱: 0.91s 

...

[COMPLETE] ● https://www.hdfc.bank.in/savings-account/super-kids-savings-account                                  | ✓ | ⏱: 4.37s 

[FETCH]... ↓ https://www.hdfc.bank.in/wholesale-banking?icid=website_target_persona_des                           | ✓ | ⏱: 3.67s 

[SCRAPE].. ◆ https://www.hdfc.bank.in/wholesale-banking?icid=website_target_persona_des                           | ✓ | ⏱: 0.51s 

[COMPLETE] ● https://www.hdfc.bank.in/wholesale-banking?icid=website_target_persona_des                           | ✓ | ⏱: 4.19s 

19 unique pages kept
 - https://www.hdfc.bank.in/
 - https://www.hdfc.bank.in/hni-banking?icid=website_target_persona_des
 - https://www.hdfc.bank.in/government-schemes?icid=website_target_persona_des
 - https://www.hdfc.bank.in/agri-banking?icid=website_target_persona_des
 - https://www.hdfc.bank.in/?icid=website_target_persona_des
 - https://www.hdfc.bank.in/msme-banking?icid=website_target_persona_des
 - https://www.hdfc.bank.in/savings-account/savings-max-account
 - https://www.hdfc.bank.in/savings-account/garv-pension-saving-account
 - https://www.hdfc.bank.in/nri-banking?icid=website_target_persona_des
 - https://www.hdfc.bank.in/savings-account
 - https://www.hdfc.bank.in/savings-account/savings-max-senior-citizen-account
 - https://www.hdfc.bank.in/savings-account/savings-max-womens-savings-account
 - https://www.hdfc.bank.in/savings-account/senior-citizen-account
 - https://www.hdfc.bank.in/savings-account/speciale-gold-womens-savings-account
 - https://www.hdfc.bank.in/saving

In [12]:
crawl4ai_documents = [
    Document(
        page_content=r.markdown.fit_markdown or r.markdown.raw_markdown,
        metadata={"source": url, "title": r.metadata.get("title", ""), "tool": "crawl4ai"},
    )
    for url, r in crawl4ai_pages.items()
]

print(len(crawl4ai_documents), "Documents ready for chunking")
print(crawl4ai_documents[0].metadata)


19 Documents ready for chunking
{'source': 'https://www.hdfc.bank.in/', 'title': 'Personal Banking & Netbanking Services | HDFC Bank', 'tool': 'crawl4ai'}


In [13]:
print(crawl4ai_documents[2].page_content)

# Government Schemes
  * Access a range of welfare programs for citizens
  * Avail benefits that support financial growth and stability
  * Simple enrolment for eligible individuals


[Atal Pension Yojana (opens in a new tab)](https://www.hdfc.bank.in/social-security-schemes/atal-pension-yojana "Atal Pension Yojana") [National Pension System (opens in a new tab)](https://www.hdfc.bank.in/nps-national-pension-system "National Pension System")
#  Types of Government Schemes 
###  Loans 
###  Insurance and Pension 
###  PMJDY 
### Features of Government Schemes:
  * **Collateral-Free Loans:** Initiatives to offer loans without the need for collateral, making credit accessible to first-time borrowers.
  * **Flexible loan amounts:** Loans range from र10,000 to र2 crore depending on the scheme and business needs.
  * **Sector-specific support:** Schemes target diverse sectors including agriculture, manufacturing, services and startups.
  * **Digital accessibility:** Many schemes are integrat

## 2. Firecrawl — the #2 pick

Firecrawl is a hosted crawling API: no browser to manage locally, built-in proxy/anti-bot
handling, and it returns clean Markdown directly from its `scrape`/`crawl` endpoints. It needs
`FIRECRAWL_API_KEY` in `.env` — the trade-off for "production-grade" is that it's a paid,
rate-limited external service rather than something that runs entirely on your machine.


In [9]:
import os

if not os.getenv("FIRECRAWL_API_KEY"):
    raise RuntimeError(
        "FIRECRAWL_API_KEY is not set. Add it to .env before running this section."
    )

from firecrawl import Firecrawl

firecrawl_client = Firecrawl(api_key=os.getenv("FIRECRAWL_API_KEY"))


ModuleNotFoundError: No module named 'firecrawl'

In [ ]:
firecrawl_home = firecrawl_client.scrape(BASE_URL, formats=["markdown"], only_main_content=True)

print(f"{len(firecrawl_home.markdown):,} chars")
print(firecrawl_home.markdown)


`only_main_content=True` is Firecrawl's equivalent of Crawl4AI's pruning filter — it's asking
the service to strip nav/boilerplate server-side rather than doing it ourselves afterward.


In [ ]:
crawl_job = firecrawl_client.crawl(
    BASE_URL,
    limit=MAX_PAGES,
    scrape_options={"formats": ["markdown"], "only_main_content": True},
)

firecrawl_pages = {
    doc.metadata.url: doc
    for doc in crawl_job.data
    if doc.metadata and doc.metadata.url and same_domain(doc.metadata.url)
}

print(f"{len(firecrawl_pages)} unique pages kept (status: {crawl_job.status})")
for url in firecrawl_pages:
    print(" -", url)


Notice `wholesale-banking/hindi-sitemap.xml` slipped into the results — Firecrawl's own link
discovery treated a sitemap file as a crawlable page. Real crawls pick up incidental non-content
URLs like this; a production pipeline would filter by content-type or path pattern (e.g. skip
anything ending in `.xml`) before handing `Document`s off to chunking.


In [ ]:
firecrawl_documents = [
    Document(
        page_content=doc.markdown or "",
        metadata={"source": url, "title": doc.metadata.title or "", "tool": "firecrawl"},
    )
    for url, doc in firecrawl_pages.items()
]

print(len(firecrawl_documents), "Documents ready for chunking")


In [ ]:
firecrawl_documents[0].metadata

In [ ]:
print(firecrawl_documents[1].page_content)

## 3. Scrapy — the #3 pick

Scrapy is the mature, general-purpose crawling framework — highly configurable and built for
scale, but it gives you raw extracted text rather than RAG-cleaned Markdown, so there's more
cleanup to do afterward. It also can't run its `CrawlerProcess` directly inside Jupyter: Scrapy
is built on the Twisted reactor, which conflicts with the asyncio event loop Jupyter already has
running. The standard workaround — used here — is to write the spider to a temporary `.py` file
and run it **out-of-process** via `scrapy runspider`, then read its JSON output back in.


In [7]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

spider_source = f'''
import scrapy

class HdfcSpider(scrapy.Spider):
    name = "hdfc_spider"
    allowed_domains = ["{ALLOWED_HOST}"]
    start_urls = ["{BASE_URL}"]

    custom_settings = {{
        "ROBOTSTXT_OBEY": True,
        "DOWNLOAD_DELAY": 1.0,
        "CLOSESPIDER_PAGECOUNT": {MAX_PAGES},
        "USER_AGENT": "{USER_AGENT}",
        "LOG_LEVEL": "INFO",
    }}

    def parse(self, response):
        title = response.css("title::text").get(default="").strip()
        text = " ".join(response.css("body ::text").getall())
        text = " ".join(text.split())
        yield {{"url": response.url, "title": title, "text": text}}

        for href in response.css("a::attr(href)").getall():
            next_url = response.urljoin(href)
            if next_url.startswith("https://{ALLOWED_HOST}"):
                yield response.follow(next_url, callback=self.parse)
'''

with tempfile.TemporaryDirectory() as tmp_dir:
    spider_path = Path(tmp_dir) / "hdfc_spider.py"
    output_path = Path(tmp_dir) / "out.json"
    spider_path.write_text(spider_source)

    subprocess.run(
        ["scrapy", "runspider", str(spider_path), "-o", str(output_path)],
        capture_output=True, text=True, timeout=120,
    )

    scrapy_items = json.loads(output_path.read_text())

print(f"{len(scrapy_items)} pages fetched (page budget was {MAX_PAGES})")
for item in scrapy_items[:MAX_PAGES]:
    print(" -", item["url"])


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [8]:
scrapy_documents = [
    Document(
        page_content=item["text"],
        metadata={"source": item["url"], "title": item["title"], "tool": "scrapy"},
    )
    for item in scrapy_items
    if same_domain(item["url"])
]

print(len(scrapy_documents), "Documents ready for chunking")
print(scrapy_documents[0].page_content[:300])


NameError: name 'scrapy_items' is not defined

Notice `scrapy_documents[0].page_content` is a long run-on string straight from `<body>` text
nodes — no Markdown structure, no boilerplate removal. That's the real difference between
"traditional" and "AI-native" crawling: Scrapy hands you everything and leaves cleanup to you;
Crawl4AI and Firecrawl do that cleanup as part of the crawl itself.


## 4. fastCRW — hosted, AI-agent-oriented (added)

`fastCRW` occupies roughly the same niche as Firecrawl: a managed cloud API (`/v1/scrape`,
`/v1/crawl`, `/v1/map`, ...) rather than something that runs entirely on your machine. It needs
`CRW_API_KEY` in `.env`, same pattern as Firecrawl above.


In [3]:
if not os.getenv("CRW_API_KEY"):
    raise RuntimeError(
        "CRW_API_KEY is not set. Add it to .env before running this section."
    )

from crw import CrwClient

crw_client = CrwClient(api_key=os.getenv("CRW_API_KEY"))


NameError: name 'os' is not defined

In [4]:
crw_home = crw_client.scrape(BASE_URL, formats=["markdown"], only_main_content=True)

print(f"{len(crw_home['markdown']):,} chars")
print(crw_home["markdown"][:800])


NameError: name 'crw_client' is not defined

In [5]:
crw_results = crw_client.crawl(BASE_URL, max_depth=1, max_pages=MAX_PAGES)

crw_pages = {}
for page in crw_results:
    page_url = (page.get("metadata") or {}).get("sourceURL")
    if page_url and same_domain(page_url) and page_url not in crw_pages:
        crw_pages[page_url] = page

print(f"{len(crw_pages)} unique pages kept")
for url in crw_pages:
    print(" -", url)


NameError: name 'crw_client' is not defined

In [6]:
crw_documents = [
    Document(
        page_content=page.get("markdown") or "",
        metadata={
            "source": url,
            "title": (page.get("metadata") or {}).get("title", ""),
            "tool": "fastcrw",
        },
    )
    for url, page in crw_pages.items()
]

print(len(crw_documents), "Documents ready for chunking")


NameError: name 'crw_pages' is not defined

In [ ]:
print(crw_documents[0].page_content)

## Summary

| Tool | Best for | RAG suitability | Setup friction | Output | Cost |
|---|---|---|---|---|---|
| **Crawl4AI** | Python + AI/RAG + dynamic websites | ⭐⭐⭐⭐⭐ | Local Playwright browser (already cached here) | Clean, pruned Markdown out of the box | Free, local compute |
| **Firecrawl** | Production-grade crawling + clean Markdown | ⭐⭐⭐⭐⭐ | Just an API key | Clean Markdown, server-side | Paid hosted API |
| **Scrapy** | Large-scale traditional crawling | ⭐⭐⭐⭐ | `pip install`, but can't run its reactor directly in Jupyter (subprocess workaround needed) | Raw extracted text — needs its own cleanup pass | Free, local compute |
| **fastCRW** | Hosted crawl/scrape/extract API for AI agents | ⭐⭐⭐⭐ | Just an API key (or self-host, unauthenticated) | Clean Markdown, server-side | Free tier (500 credits), then paid |

All four end up producing the same thing: a list of `Document` objects with `page_content`
and `metadata={"source": ..., "title": ..., "tool": ...}` — exactly what
`l22_document_loaders.ipynb` and `l22_chunking_strategies.ipynb` expect as input. Crawling is
the step that gets you *to* a `Document`; everything downstream (chunking, embedding,
retrieval) doesn't care which of the four tools produced it.


## Takeaways

- Crawling is a distinct pipeline stage from loading — loaders assume a file already exists;
  crawlers are what get you that file from a live site in the first place.
- Crawl4AI, Firecrawl, and fastCRW all do RAG-oriented cleanup (pruned/main-content Markdown) as
  part of the crawl — Scrapy gives you raw text and expects a separate cleaning + chunking step.
- The "AI/RAG suitability" difference between these tools mostly comes down to *how much
  boilerplate you have to strip yourself* before the content is embedding-ready.
- Firecrawl and fastCRW solve the same problem (hosted, agent-oriented crawling) — which one to
  use in practice comes down to pricing, rate limits, and how much you trust a newer entrant
  versus a more established service, not a fundamental capability gap.
- Respecting `robots.txt`, staying same-domain, and keeping a small page budget isn't just
  politeness — it's what keeps an educational crawl from ever looking like an attack on a real
  production site.
